## 1. Connecting to the Database

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [3]:
from src.adx_connection import client, database

In [4]:
query = "NIFTY50_all | take 10"
response = client.execute(database, query)

for row in response.primary_results[0]:
    print(row)

['2007-11-27 00:00:00+00:00', 'MUNDRAPORT', 'EQ', '440.0', '770.0', '1050.0', '770.0', '959.0', '962.9', '984.72', '27294366', '2687719053785000.0', 'None', '9859619.0', '0.3612']
['2007-11-28 00:00:00+00:00', 'MUNDRAPORT', 'EQ', '962.9', '984.0', '990.0', '874.0', '885.0', '893.9', '941.38', '4581338', '431276530165000.0', 'None', '1453278.0', '0.3172']
['2007-11-29 00:00:00+00:00', 'MUNDRAPORT', 'EQ', '893.9', '909.0', '914.75', '841.0', '887.0', '884.2', '888.09', '5124121', '455065846265000.0', 'None', '1069678.0', '0.2088']
['2007-11-30 00:00:00+00:00', 'MUNDRAPORT', 'EQ', '884.2', '890.0', '958.0', '890.0', '929.0', '921.55', '929.17', '4609762', '428325662830000.0', 'None', '1260913.0', '0.2735']
['2007-12-03 00:00:00+00:00', 'MUNDRAPORT', 'EQ', '921.55', '939.75', '995.0', '922.0', '980.0', '969.3', '965.65', '2977470', '287519974300000.0', 'None', '816123.0', '0.2741']
['2007-12-04 00:00:00+00:00', 'MUNDRAPORT', 'EQ', '969.3', '985.0', '1056.0', '976.0', '1049.0', '1041.45', '

In [4]:
query = "NIFTY50_all |count"
response = client.execute(database, query)

for row in response.primary_results[0]:
    print(row)

['235192']


## 2. Converting KQL query result to a pandas DataFrame

In [12]:
query = "NIFTY50_all"
response = client.execute(database, query)

from azure.kusto.data.helpers import dataframe_from_result_table
df_result = dataframe_from_result_table(response.primary_results[0])
df_result.head()

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,_Deliverble
0,2007-11-27 00:00:00+00:00,MUNDRAPORT,EQ,440.0,770.0,1050.0,770.0,959.0,962.9,984.72,27294366,2687719053785000.0,<NA>,9859619.0,0.3612
1,2007-11-28 00:00:00+00:00,MUNDRAPORT,EQ,962.9,984.0,990.0,874.0,885.0,893.9,941.38,4581338,431276530165000.0,<NA>,1453278.0,0.3172
2,2007-11-29 00:00:00+00:00,MUNDRAPORT,EQ,893.9,909.0,914.75,841.0,887.0,884.2,888.09,5124121,455065846265000.0,<NA>,1069678.0,0.2088
3,2007-11-30 00:00:00+00:00,MUNDRAPORT,EQ,884.2,890.0,958.0,890.0,929.0,921.55,929.17,4609762,428325662830000.0,<NA>,1260913.0,0.2735
4,2007-12-03 00:00:00+00:00,MUNDRAPORT,EQ,921.55,939.75,995.0,922.0,980.0,969.3,965.65,2977470,287519974300000.0,<NA>,816123.0,0.2741


In [6]:
# Helper fn
def run_kql(query):
    """Execute a KQL query and return the result as a pandas DataFrame."""
    response = client.execute(database, query)
    return dataframe_from_result_table(response.primary_results[0])

## 3. Data Cleaning

In [7]:
# Testing Turnover fix on a single stock before applying broadly
run_kql("""
    NIFTY50_all
    | where Symbol == 'RELIANCE'
    | extend  Turnover_corrected = Turnover / 100000
    | project Date, Turnover, Turnover_corrected
    | take 10
        """)

,Date,Turnover,Turnover_corrected
0,2000-01-03 00:00:00+00:00,111131914445000.0,1111319144.45
1,2000-01-04 00:00:00+00:00,250022190690000.0,2500221906.9
2,2000-01-05 00:00:00+00:00,737369727570000.0,7373697275.7
3,2000-01-06 00:00:00+00:00,463325404090000.0,4633254040.9
4,2000-01-07 00:00:00+00:00,613838765740000.0,6138387657.4
5,2000-01-10 00:00:00+00:00,419085494795000.0,4190854947.95
6,2000-01-11 00:00:00+00:00,371819207090000.0,3718192070.9
7,2000-01-12 00:00:00+00:00,356708113910000.0,3567081139.1
8,2000-01-13 00:00:00+00:00,532418081645000.0,5324180816.45
9,2000-01-14 00:00:00+00:00,425587852020000.0,4255878520.2


##### Confirmed: Turnover_Corrected matches implied turnover from earlier pandas check (~1% difference, expected)

In [38]:
# Pulling the full cleaned dataset (all stocks) with Turnover_corrected added, 
# keeping the original Turnover column for transparency/comparison
cleaned_data = run_kql(
"""
    NIFTY50_all
    | extend Turnover_corrected = Turnover / 100000
"""
)

cleaned_data.head(5)

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,_Deliverble,Turnover_corrected
0,2007-11-27 00:00:00+00:00,MUNDRAPORT,EQ,440.0,770.0,1050.0,770.0,959.0,962.9,984.72,27294366,2687719053785000.0,<NA>,9859619.0,0.3612,26877190537.849998
1,2007-11-28 00:00:00+00:00,MUNDRAPORT,EQ,962.9,984.0,990.0,874.0,885.0,893.9,941.38,4581338,431276530165000.0,<NA>,1453278.0,0.3172,4312765301.65
2,2007-11-29 00:00:00+00:00,MUNDRAPORT,EQ,893.9,909.0,914.75,841.0,887.0,884.2,888.09,5124121,455065846265000.0,<NA>,1069678.0,0.2088,4550658462.65
3,2007-11-30 00:00:00+00:00,MUNDRAPORT,EQ,884.2,890.0,958.0,890.0,929.0,921.55,929.17,4609762,428325662830000.0,<NA>,1260913.0,0.2735,4283256628.3
4,2007-12-03 00:00:00+00:00,MUNDRAPORT,EQ,921.55,939.75,995.0,922.0,980.0,969.3,965.65,2977470,287519974300000.0,<NA>,816123.0,0.2741,2875199743.0


In [39]:
cleaned_data.shape

(235192, 16)

In [40]:
# Sanity check: confirming dtypes and null counts match expectations after 
print(cleaned_data.dtypes)
print(cleaned_data.isna().sum())

Date                  datetime64[us, UTC]
Symbol                                str
Series                                str
Prev Close                        Float64
Open                              Float64
High                              Float64
Low                               Float64
Last                              Float64
Close                             Float64
VWAP                              Float64
Volume                              Int64
Turnover                          Float64
Trades                            Float64
Deliverable Volume                Float64
_Deliverble                       Float64
Turnover_corrected                Float64
dtype: object
Date                       0
Symbol                     0
Series                     0
Prev Close                 0
Open                       0
High                       0
Low                        0
Last                       0
Close                      0
VWAP                       0
Volume                  

### Data Cleaning: Summary of Key Findings

- **Turnover was inflated by ~100,000x** across all stocks (verified on RELIANCE, TCS via 
  Volume × Close comparison), likely a ₹Lakhs-to-rupees conversion error in the source. 
  Fixed via `Turnover_corrected = Turnover / 100000`, applied in KQL at query time.

- **Trades is genuinely missing (not broken) for 2000–2010** — 100% missing before 2011, 
  transitioning that year, fully populated from 2012 onward. Reflects a reporting cutoff 
  by NSE, not a data error. Left as NaN (not imputed); any Trades-based analysis will be 
  scoped to 2012+.

- **No duplicate records** — zero duplicates on (Date, Symbol), confirming this pair as the 
  natural unique key for the dataset (235,192 rows, 65 symbols, 2000–2021).

In [41]:
cleaned_data[
    (cleaned_data["Symbol"] == "BHARTI") &
    (cleaned_data["Date"].between("2002-02-15", "2002-02-20"))
][["Date", "Symbol", "Prev Close", "Open", "High", "Low", "Close", "Volume"]]

,Date,Symbol,Prev Close,Open,High,Low,Close,Volume
25572,2002-02-18 00:00:00+00:00,BHARTI,0.0,51.9,51.9,43.5,44.35,10381190
25573,2002-02-19 00:00:00+00:00,BHARTI,44.35,45.15,45.15,41.3,41.7,3552660
25574,2002-02-20 00:00:00+00:00,BHARTI,41.7,40.6,42.8,40.25,41.25,2512964


In [42]:
import numpy as np

cleaned_data.loc[
    (cleaned_data["Prev Close"] == 0),
    ["Prev Close"]
] = np.nan

### Data Quality Finding: Zero Price Values
- One record (BHARTI, 18 February 2002) contained zero values in Prev Close, which is invalid price observation given the other positive prices recorded for that day. That value was converted to NaN. The record was retained to preserve valid information in the remaining columns. No imputation was performed.

In [43]:
# Saving the cleaned dataset as a Parquet checkpoint for the EDA notebook
cleaned_data.to_parquet("../data/nifty50_cleaned.parquet", index=False)